# 02_pipeline — config-driven orchestration template

Build, check, publish, and record evidence for governed Fabric data pipelines.

This notebook is intentionally thin and beginner friendly. The default happy path reads two smoke source Lakehouse tables, transforms them into two Lakehouse target DataFrames, writes both targets to the `SmokeTest` schema, and records lineage plus run-summary evidence.

To adapt the template, edit the clearly marked **USER EDIT SECTION** blocks. The main business-logic edit area is **Step 8**. Add source DataFrame reads before `SOURCE_TABLES`, keep each table's guardrails beside that table config, add transformations before `TARGET_TABLES`, and then add target dictionaries for DataFrames that already exist. Do not copy profiling, schema, freshness, profile behavior, DQ, catalogue-evidence, lineage, or runtime-summary orchestration code.

FabricOps enriches source and target entries before running profiling, schema validation, profile behavior enforcement, DQ enforcement, catalogue evidence, Lakehouse writes, lineage, and runtime summary from the config lists.


## 1. Run `00_env_config`

Load the shared FabricOps environment, path configuration, sample metadata, and metadata lakehouse routing.


In [ ]:
%run 00_env_config


## 2. Import required functions

The notebook imports existing FabricOps callables for agreement selection, table-config preparation, guardrail orchestration, explicit target writes, lineage, and runtime-summary evidence.


In [ ]:
from pyspark.sql import functions as F

from fabricops_kit.config import _current_audit_timestamp

from fabricops_kit import (
    get_selected_agreement,
    prepare_pipeline_table_configs,
    read_lakehouse_csv,
    read_lakehouse_excel,
    read_lakehouse_parquet,
    read_lakehouse_table,
    read_warehouse_table,
    run_table_guardrails,
    widget_select_agreement,
    write_lakehouse_table,
    write_pipeline_lineage,
    write_pipeline_run_summary,
    write_warehouse_table,
)


## 3. Select data agreement and capture run context

Select the agreement that this pipeline satisfies. The selector registers this notebook in `METADATA_NOTEBOOK_REGISTRY` using the metadata target configured by `00_env_config`. The run context values are reused by guardrail evidence, lineage, and runtime summary writes.


In [ ]:
PIPELINE_STARTED_AT = _current_audit_timestamp(config=CONFIG)
RUN_ID = RUN_CONTEXT.run_id
ENV_NAME = ENV
PIPELINE_NAME = RUN_CONTEXT.runtime_metadata.get("currentNotebookName", "02_pipeline")

widget_select_agreement(
    CONFIG,
    env_name=ENV_NAME,
    spark_session=spark,
    metadata_schema=METADATA_SCHEMA,
    register_notebook=True,
    notebook_type="02_pipeline",
    pipeline_name=PIPELINE_NAME,
)

AGREEMENT = get_selected_agreement()
AGREEMENT_ID = AGREEMENT.get("agreement_id", "")
AGREEMENT_CONTRACT_VERSION = AGREEMENT.get("agreement_contract_version", AGREEMENT.get("contract_version", ""))
NOTEBOOK_REGISTRY_ID = AGREEMENT.get("notebook_registry_id", AGREEMENT.get("registration_id", ""))
NOTEBOOK_ID = AGREEMENT.get("notebook_id", RUN_CONTEXT.runtime_metadata.get("currentNotebookId", ""))


---

# SOURCE AREA — Steps 4 to 7

Read the default two source tables, configure their guardrails, optionally inspect schemas, and validate sources before transformation.


## 4. USER EDIT SECTION — read source DataFrames

Read each source DataFrame first, using the existing FabricOps IO helper that matches where the data lives. The default demo reads two smoke source tables created by `example_pipeline_smoke_test.ipynb` so users can see a many-source pipeline before configuring guardrails or targets.

Optional CSV, parquet, Excel, warehouse, and Spark table examples are shown as commented alternatives. Keep the primary path concrete: read DataFrames here, transform them later, and only declare target tables after target DataFrames exist.


In [ ]:
df_orders = read_lakehouse_table(
    CONFIG,
    ENV_NAME,
    "source",
    "smoke_src_orders_happy",
    schema="SmokeTest",
    spark_session=spark,
)

df_customers = read_lakehouse_table(
    CONFIG,
    ENV_NAME,
    "source",
    "smoke_src_customers_happy",
    schema="SmokeTest",
    spark_session=spark,
)

# CSV file in a configured Lakehouse target:
# df_orders = read_lakehouse_csv(
#     CONFIG,
#     ENV_NAME,
#     "source",
#     "path/to/orders.csv",
#     spark_session=spark,
#     header=True,
# )

# Parquet file or folder in a configured Lakehouse target:
# df_orders = read_lakehouse_parquet(
#     CONFIG,
#     ENV_NAME,
#     "source",
#     "path/to/orders.parquet",
#     verbose=True,
#     spark_session=spark,
# )

# Excel file in a configured Lakehouse target:
# df_customers = read_lakehouse_excel(
#     CONFIG,
#     ENV_NAME,
#     "source",
#     "path/to/customers.xlsx",
#     sheet_name=0,
#     spark_session=spark,
# )

# Warehouse table:
# df_orders = read_warehouse_table(
#     CONFIG,
#     ENV_NAME,
#     "source",
#     "dbo",
#     "orders",
#     spark_session=spark,
# )

# Custom Spark table reference:
# df_customers = spark.read.table("database.customers")


## 5. USER EDIT SECTION — configure source tables and source guardrails

Configure each already-read source DataFrame together with the guardrails and catalogue evidence that belong to that specific source. FabricOps derives governance `dataset_name` from `table_name` and uses `layer` as the default governance `stage`, so normal source examples do not need `dataset_name`.

Valid FabricOps layer/stage concepts:

- `source` = raw/source lakehouse table.
- `unified` = cleaned/conformed lakehouse table.
- `product` = curated product or warehouse output.
- `metadata` = governance evidence lakehouse. It is normally configured in `00_env_config` and is not usually selected as a business source table.

To add sources, read another DataFrame above and add another dictionary to `SOURCE_TABLES` with a unique `key` and that source's own schema, freshness, DQ, profile, and expected-schema settings. Do not copy profiling, schema, freshness, profile behavior, DQ, or catalogue-evidence code.

Advanced override support: add `dataset_name` or `stage` inside a specific source table config only when that table needs a governance override.


In [ ]:
SOURCE_TABLES = [
    {
        "key": "orders",
        "df": df_orders,
        "layer": "source",
        "table_name": "smoke_src_orders_happy",
        "watermark_column": "order_date",
        "schema_preset": "allow_new_columns",
        "load_behavior": "append",
        "freshness_column": "order_date",
        "freshness_max_lag_days": 1,
        "freshness_severity": "blocking",
        "dq_preset": "approved_rules",
        "distribution_columns": ["status", "order_amount", "country_code"],
        "exclude_columns": None,
        "expected_schema": {
            "order_id": "bigint",
            "customer_id": "bigint",
            "order_date": "date",
            "ingestion_ts": "timestamp",
            "status": "string",
            "order_amount": "double",
            "country_code": "string",
        },
    },
    {
        "key": "customers",
        "df": df_customers,
        "layer": "source",
        "table_name": "smoke_src_customers_happy",
        "watermark_column": "effective_date",
        "schema_preset": "allow_new_columns",
        "load_behavior": "append",
        "freshness_column": "effective_date",
        "freshness_max_lag_days": 1,
        "freshness_severity": "blocking",
        "dq_preset": "skip",
        "distribution_columns": ["customer_segment", "customer_country_code"],
        "exclude_columns": None,
        "expected_schema": {
            "customer_id": "bigint",
            "customer_name": "string",
            "customer_segment": "string",
            "customer_country_code": "string",
            "effective_date": "date",
            "ingestion_ts": "timestamp",
        },
    },
]

SOURCE_TABLES, SOURCE_CONFIG_BY_KEY = prepare_pipeline_table_configs(
    SOURCE_TABLES,
    {},
    table_role="source",
)

# Convenience aliases keep the demo transformations easy to read.
df_orders = SOURCE_CONFIG_BY_KEY["orders"]["df"]
df_customers = SOURCE_CONFIG_BY_KEY["customers"]["df"]

# Optional advanced per-table governance overrides, only when needed:
# "dataset_name": "governance_dataset_override",
# "stage": "source",


## 6. Optional: inspect source schemas

Use this optional authoring check before editing `expected_schema` values for one or more sources.


In [ ]:
# Optional authoring check: inspect Spark schemas before writing expected_schema.
for source in SOURCE_TABLES:
    print(f"Schema for {source['key']} ({source.get('table_name', '')})")
    source["df"].printSchema()


## 7. Run source guardrails before transformation

FabricOps runs profiling, schema validation, profile behavior checks, DQ checks, catalogue evidence, and optional guardrail stopping through `run_table_guardrails`. Before catalogue evidence is written to `METADATA_DATA_CATALOGUE`, FabricOps schema-aligns generated evidence columns such as row counts, DQ counts, percentages, timestamps, and booleans to the metadata table schema. Source guardrails run before transformation, and most users should not need to customize this orchestration code.


In [ ]:
source_guardrail_results = run_table_guardrails(
    SOURCE_TABLES,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    spark_session=spark,
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
    stop_on_failure=True,
)

display(source_guardrail_results["summary"])

# Runtime summary and lineage cells reuse these package-generated evidence objects.
source_schema_results = source_guardrail_results["schema_results"]
source_freshness_results = source_guardrail_results["freshness_results"]
source_stability_results = source_guardrail_results["stability_results"]
source_dq_results = source_guardrail_results["dq_results"]
source_catalogue_status = source_guardrail_results["catalogue_status"]
source_evidence_definitions = source_guardrail_results["evidence_definitions"]


---

# TRANSFORMATION AREA — Step 8

This is the main user edit area for business transformation logic.


## 8. USER EDIT SECTION — main business transformation logic

**Most users should make their business logic changes here.** Create one target DataFrame for each target table you plan to publish. The default happy path creates exactly two target DataFrames:

- `df_orders_enriched`
- `df_orders_summary`

FabricOps guardrails, audit columns, Lakehouse writes, lineage, and run-summary evidence are handled in later sections.

The demo keeps geography explicit: `orders.country_code` is transaction/order geography, while `customers.customer_country_code` is customer-profile geography. The join selects both columns by distinct names so the enriched target has no duplicate `country_code` columns. Summaries group by the order-level `country_code`.


In [ ]:
df_orders_enriched = (
    df_orders.alias("orders")
    .join(df_customers.alias("customers"), on="customer_id", how="left")
    .select(
        F.col("orders.order_id"),
        F.col("orders.customer_id"),
        F.col("customers.customer_name"),
        F.col("customers.customer_segment"),
        F.col("orders.country_code"),
        F.col("customers.customer_country_code"),
        F.col("orders.order_date"),
        F.col("orders.ingestion_ts"),
        F.col("orders.status"),
        F.col("orders.order_amount"),
    )
    .withColumn(
        "order_amount_band",
        F.when(F.col("order_amount") >= F.lit(100), F.lit("high"))
        .when(F.col("order_amount") >= F.lit(25), F.lit("medium"))
        .otherwise(F.lit("low")),
    )
)

df_orders_summary = (
    df_orders_enriched
    .groupBy("customer_segment", "country_code")
    .agg(
        F.count("order_id").alias("order_count"),
        F.sum("order_amount").alias("total_order_amount"),
        F.max("order_date").alias("latest_order_date"),
    )
)


---

# TARGET AREA — Steps 9 to 14

Configure two Lakehouse targets, run target guardrails, write the targets, then record lineage and run-summary evidence. Warehouse writes are not part of the default happy path.


## 9. USER EDIT SECTION — configure target tables and target guardrails

Declare target table configs only after the transformed target DataFrames exist, with each target's own schema, freshness, DQ, profile, and expected-schema settings. The default demo intentionally uses two Lakehouse targets in the `unified` layer so the starter pipeline can run without warehouse write permissions.

To add targets, create another DataFrame in the transform section, then add another dictionary to `TARGET_TABLES` with a unique `key`. Because Step 11 uses beginner-friendly explicit writes instead of a generic loop, also add a matching explicit write call in Step 11 for every new target key. Do not copy profiling, schema, freshness, profile behavior, DQ, catalogue-evidence, lineage, or runtime-summary orchestration code.

Advanced override support: add `dataset_name`, `stage`, `target_layer`, `target_name`, `dq_preset`, or `partition_by` inside a specific target table config only when that table needs to differ from the guardrail or write defaults.


In [ ]:
TARGET_TABLES = [
    {
        "key": "orders_enriched",
        "df": df_orders_enriched,
        "layer": "unified",
        "table_name": "smoke_unified_orders_enriched",
        "write_mode": "overwrite",
        "watermark_column": "order_date",
        "schema": "SmokeTest",
        "schema_preset": "strict",
        "load_behavior": "overwrite",
        "freshness_column": "order_date",
        "freshness_max_lag_days": 1,
        "freshness_severity": "blocking",
        "dq_preset": "approved_rules",
        "distribution_columns": ["status", "order_amount", "country_code"],
        "exclude_columns": None,
        "expected_schema": {
            "order_id": "bigint",
            "customer_id": "bigint",
            "customer_name": "string",
            "customer_segment": "string",
            "country_code": "string",
            "customer_country_code": "string",
            "order_date": "date",
            "ingestion_ts": "timestamp",
            "status": "string",
            "order_amount": "double",
            "order_amount_band": "string",
            "_fabricops_run_id": "string",
            "_fabricops_pipeline_name": "string",
            "_fabricops_created_at": "string",
        },
    },
    {
        "key": "orders_summary",
        "df": df_orders_summary,
        "layer": "unified",
        "table_name": "smoke_unified_orders_summary",
        "write_mode": "overwrite",
        "watermark_column": "latest_order_date",
        "schema": "SmokeTest",
        "schema_preset": "strict",
        "load_behavior": "overwrite",
        "freshness_column": "latest_order_date",
        "freshness_max_lag_days": 1,
        "freshness_severity": "blocking",
        "dq_preset": "skip",
        "distribution_columns": ["customer_segment", "country_code"],
        "exclude_columns": None,
        "expected_schema": {
            "customer_segment": "string",
            "country_code": "string",
            "order_count": "bigint",
            "total_order_amount": "double",
            "latest_order_date": "date",
            "_fabricops_run_id": "string",
            "_fabricops_pipeline_name": "string",
            "_fabricops_created_at": "string",
        },
    },
]

# Optional advanced per-table overrides, only when needed:
# "dataset_name": "governance_dataset_override",
# "stage": "product",
# "target_layer": "product",
# "target_name": "written_table_name_override",
# "dq_preset": "approved_rules",
# "partition_by": ["order_date"],
# "repartition_by": ["customer_id"],
# "options": {"overwriteSchema": "true"},

TARGET_TABLES, TARGET_CONFIG_BY_KEY = prepare_pipeline_table_configs(
    TARGET_TABLES,
    {},
    table_role="target",
    run_id=RUN_ID,
    pipeline_name=PIPELINE_NAME,
)

# Convenience alias keeps the default lineage dataset name easy to read.
PRIMARY_TARGET_CONFIG = TARGET_CONFIG_BY_KEY["orders_enriched"]


## 10. Run target guardrails before writes

Target profiling, schema validation, profile behavior checks, DQ checks, and catalogue evidence run for every config in `TARGET_TABLES`. Target writes do not happen unless `run_table_guardrails(..., stop_on_failure=True)` completes.


In [ ]:
target_guardrail_results = run_table_guardrails(
    TARGET_TABLES,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    spark_session=spark,
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
    stop_on_failure=True,
)

display(target_guardrail_results["summary"])

target_schema_results = target_guardrail_results["schema_results"]
target_freshness_results = target_guardrail_results["freshness_results"]
target_stability_results = target_guardrail_results["stability_results"]
target_dq_results = target_guardrail_results["dq_results"]
target_catalogue_status = target_guardrail_results["catalogue_status"]
target_evidence_definitions = target_guardrail_results["evidence_definitions"]


## 11. Write target Lakehouse tables

Only after all configured target guardrails pass, explicitly write each prepared target config DataFrame to its Lakehouse table. The prepared configs in `TARGET_CONFIG_BY_KEY` include FabricOps audit columns and write metadata, so the published tables match the DataFrames validated by Step 10. The default happy path writes two `SmokeTest` tables and does not require warehouse write permissions.

If you add another target to `TARGET_TABLES`, also add a matching explicit write call in this section for that target key.

Warehouse writes may require additional workspace or item permissions, so they are shown only as a commented optional example and should not block the starter demo.


In [ ]:
target_write_options = {"overwriteSchema": "true"}

orders_enriched_target = TARGET_CONFIG_BY_KEY["orders_enriched"]
orders_summary_target = TARGET_CONFIG_BY_KEY["orders_summary"]

target_write_status = {}

write_lakehouse_table(
    orders_enriched_target["df"],
    CONFIG,
    ENV_NAME,
    orders_enriched_target["target_layer"],
    orders_enriched_target["target_name"],
    schema=orders_enriched_target.get("schema"),
    mode=orders_enriched_target.get("write_mode", "overwrite"),
    partition_by=orders_enriched_target.get("partition_by"),
    repartition_by=orders_enriched_target.get("repartition_by"),
    options=orders_enriched_target.get("options", target_write_options),
)
target_write_status["orders_enriched"] = f"written: SmokeTest.{orders_enriched_target['target_name']}"

write_lakehouse_table(
    orders_summary_target["df"],
    CONFIG,
    ENV_NAME,
    orders_summary_target["target_layer"],
    orders_summary_target["target_name"],
    schema=orders_summary_target.get("schema"),
    mode=orders_summary_target.get("write_mode", "overwrite"),
    partition_by=orders_summary_target.get("partition_by"),
    repartition_by=orders_summary_target.get("repartition_by"),
    options=orders_summary_target.get("options", target_write_options),
)
target_write_status["orders_summary"] = f"written: SmokeTest.{orders_summary_target['target_name']}"

# Optional warehouse example (not part of the default happy path):
# Warehouse writes may require additional permissions and should not block the starter demo.
# write_warehouse_table(
#     orders_summary_target["df"],
#     CONFIG,
#     ENV_NAME,
#     "product",
#     "dbo",
#     "smoke_product_orders_summary",
#     mode="overwrite",
# )

display(target_write_status)


## 12. USER EDIT SECTION — lineage relationships

Describe how configured source tables produce configured target tables. Each relationship uses source and target keys from `SOURCE_TABLES` and `TARGET_TABLES`.

A relationship can contain one or more source keys and one or more target keys. FabricOps expands many-to-many relationships into table-level lineage rows for every source-target pair.


In [ ]:
LINEAGE_RELATIONSHIPS = [
    {
        "sources": ["orders", "customers"],
        "targets": ["orders_enriched", "orders_summary"],
        "operation": "join orders to customers, enrich orders, and summarize by customer attributes",
        "description": (
            f"{SOURCE_CONFIG_BY_KEY['orders']['table_name']} and "
            f"{SOURCE_CONFIG_BY_KEY['customers']['table_name']} produce "
            f"{TARGET_CONFIG_BY_KEY['orders_enriched']['table_name']} and "
            f"{TARGET_CONFIG_BY_KEY['orders_summary']['table_name']}."
        ),
    },
]


## 13. Write lineage

FabricOps writes lineage evidence after target writes complete.


In [ ]:
lineage_result = write_pipeline_lineage(
    spark=spark,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    source_definitions=source_evidence_definitions,
    target_definitions=target_evidence_definitions,
    relationships=LINEAGE_RELATIONSHIPS,
    dataset_name=PRIMARY_TARGET_CONFIG["dataset_name"],
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
)


## 14. Write runtime summary

Runtime evidence is stored in `METADATA_PIPELINE_RUNS` and displayed for operational support.


In [ ]:
catalogue_status = "written" if source_catalogue_status and target_catalogue_status else "not_written"

run_summary = write_pipeline_run_summary(
    spark=spark,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
    started_at=PIPELINE_STARTED_AT,
    completed_at=_current_audit_timestamp(config=CONFIG),
    status="completed",
    source_definitions=source_evidence_definitions,
    target_definitions=target_evidence_definitions,
    source_schema_results=source_schema_results,
    target_schema_results=target_schema_results,
    source_freshness_results=source_freshness_results,
    target_freshness_results=target_freshness_results,
    source_stability_results=source_stability_results,
    target_stability_results=target_stability_results,
    source_dq_results=source_dq_results,
    target_dq_results=target_dq_results,
    lineage_status=lineage_result.get("status", "unknown"),
    catalogue_status=catalogue_status,
    message="Pipeline completed and metadata evidence was written.",
)

display(run_summary)
